# 03_gnn_text_v2 — SOTA Traffic GNN (Qwen3.5)

Thin notebook: all components live in `src/model_v2.py`, `src/train.py`.

Upgrades over v1:
- **GatedTCN** — gated dilated convolutions (fixed truncation)
- **DiffusionGCN** — K-hop spatial aggregation (fixed extra hop)
- **AdaptiveAdj** — learned adjacency mixed with static road network
- **CrossModalFusion** — multi-head cross-attention over text tokens
- **Qwen3.5-0.8B** — token-level LLM text encoder (frozen + LoRA, fixed speed)
- **Gradient clipping + LR scheduler** — stable training
- **Auto-caching** — embeddings recompute only when texts change

## Section 1: Setup & Data

In [ ]:
import sys
sys.path.insert(0, "..")

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [ ]:
from src import (
    load_train_speeds, load_train_texts,
    load_adjacency,
    build_windows, compute_norm_stats, normalize, denormalize,
    train_val_split, compute_mse,
    cache_embeddings,
)
from src.model_v2 import SOTATrafficGNN
from src.train import (
    build_adj_tensor,
    encode_texts_qwen, load_qwen,
    train_one_config,
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:
s1, s2 = load_train_speeds()
t1, t2 = load_train_texts()
adj_raw = load_adjacency()

X1, T1, Y1 = build_windows(s1, t1)
X2, T2, Y2 = build_windows(s2, t2)

X_all = np.concatenate([X1, X2], axis=0).astype(np.float32)
Y_all = np.concatenate([Y1, Y2], axis=0).astype(np.float32)
T_all = np.array(T1 + T2)

mean, std = compute_norm_stats(np.concatenate([s1, s2], axis=0))
X_norm = normalize(X_all, mean, std)
Y_norm = normalize(Y_all, mean, std)

X_tr, _, Y_tr, X_va, _, Y_va = train_val_split(X_norm, None, Y_norm, val_frac=0.2)
T_tr = T_all[:len(X_tr)]
T_va = T_all[len(X_tr):]

adj_t = build_adj_tensor(adj_raw).to(DEVICE)

print(f"Train: {X_tr.shape}, Val: {X_va.shape}")
print(f"Adj: {adj_t.shape}, nnz={(adj_t > 0).sum().item()}")

## Section 2: Qwen3.5 Text Encoder (auto-cached)

Qwen3.5-0.8B (1024-dim token-level). Loads directly to GPU via `device_map`. Cached with content-hash — recompute only when train/val texts change.

In [ ]:
qwen_model, qwen_tokenizer = load_qwen("Qwen/Qwen3.5-0.8B", device=DEVICE)

def encode_qwen_wrapper(texts, batch_size=32):
    return encode_texts_qwen(texts, qwen_model, qwen_tokenizer, DEVICE,
                             batch_size=batch_size)

T_emb_tr, T_emb_va = cache_embeddings(
    T_tr.tolist(), T_va.tolist(),
    encoder_fn=encode_qwen_wrapper,
    cache_name="qwen",
    encoder_kwargs={"batch_size": 32},
)
print(f"Qwen3.5: train={T_emb_tr.shape}, val={T_emb_va.shape}")

In [ ]:
class TrafficDS(Dataset):
    def __init__(self, X, T_emb, Y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.T = torch.tensor(T_emb, dtype=torch.float32)
        self.Y = torch.tensor(Y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        return self.X[i], self.T[i], self.Y[i]


def make_loaders(T_emb_tr, T_emb_va, batch_size=32):
    tr = TrafficDS(X_tr, T_emb_tr, Y_tr)
    va = TrafficDS(X_va, T_emb_va, Y_va)
    return (
        DataLoader(tr, batch_size=batch_size, shuffle=True, pin_memory=True),
        DataLoader(va, batch_size=batch_size, shuffle=False, pin_memory=True),
    )

print("DataLoaders ready.")

## Section 3: Smoke Test

In [ ]:
x_dummy = torch.randn(2, 15, 1260)
adj_dummy = torch.eye(1260) * 0.5

for name, text_dim, text_shape, use_text in [
    ("Qwen cross-attn", 1024, (2, 32, 1024), True),
    ("Speed-only", 384, (2, 384), False),
]:
    model = SOTATrafficGNN(
        d_model=64, num_tcn_layers=4, num_gcn_layers=4,
        num_hops=2, text_dim=text_dim, n_heads=4, dropout=0.2,
        use_text=use_text, fusion="cross", use_adaptive_adj=True,
    )
    t = torch.randn(text_shape)
    with torch.no_grad():
        out = model(x_dummy, t, adj_dummy)
    n = sum(p.numel() for p in model.parameters())
    print(f"  {name:<20} params={n:>8,}  out={list(out.shape)}")

print("All variants OK.")

## Section 4: Train & Evaluate

In [ ]:
@torch.no_grad()
def eval_mse(model, loader, adj, device):
    model.eval()
    preds, targets = [], []
    for Xb, Tb, Yb in loader:
        Xb, Tb, Yb = Xb.to(device), Tb.to(device), Yb.to(device)
        pb = model(Xb, Tb, adj)
        preds.append(pb.cpu().numpy())
        targets.append(Yb.cpu().numpy())
    yp = denormalize(np.concatenate(preds, axis=0), mean, std)
    yt = denormalize(np.concatenate(targets, axis=0), mean, std)
    return compute_mse(yp, yt)

print("Evaluation helper ready.")

In [ ]:
model = SOTATrafficGNN(
    d_model=64, num_tcn_layers=4, num_gcn_layers=4,
    num_hops=2, text_dim=1024, n_heads=4, dropout=0.2,
    use_text=True, fusion="cross", use_adaptive_adj=True,
).to(DEVICE)

tl, vl = make_loaders(T_emb_tr, T_emb_va, batch_size=16)
best_val_norm, n_params, n_epochs = train_one_config(
    model, tl, vl, adj_t, DEVICE, epochs=80, lr=1e-3, patience=15,
)

mse = eval_mse(model, vl, adj_t, DEVICE)
print(f"Val MSE: {mse:.4f}  ({n_params:,} params, {n_epochs} epochs)")

for hi, hn in enumerate(["h5", "h10", "h15"]):
    # Predict on val set per horizon
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for Xb, Tb, Yb in vl:
            Xb, Tb, Yb = Xb.to(DEVICE), Tb.to(DEVICE), Yb.to(DEVICE)
            pb = model(Xb, Tb, adj_t)
            preds.append(pb.cpu().numpy())
            targets.append(Yb.cpu().numpy())
    yp = denormalize(np.concatenate(preds, axis=0), mean, std)
    yt = denormalize(np.concatenate(targets, axis=0), mean, std)
    mse_h = compute_mse(yp[:, hi:hi+1, :], yt[:, hi:hi+1, :])
    print(f"  {hn}: {mse_h:.4f}")

## Section 5: Retrain on All Data & Submit

In [ ]:
T_emb_all = encode_texts_qwen(T_all.tolist(), qwen_model, qwen_tokenizer,
                               DEVICE, batch_size=32)
print(f"Full train embeddings: {T_emb_all.shape}")

full_ds = TrafficDS(X_norm, T_emb_all, Y_norm)
full_loader = DataLoader(full_ds, batch_size=16, shuffle=True)

model_full = SOTATrafficGNN(
    d_model=64, num_tcn_layers=4, num_gcn_layers=4,
    num_hops=2, text_dim=1024, n_heads=4, dropout=0.2,
    use_text=True, fusion="cross", use_adaptive_adj=True,
).to(DEVICE)

opt = torch.optim.Adam(model_full.parameters(), lr=1e-3)
crit = nn.MSELoss()
for epoch in range(1, 61):
    model_full.train()
    total = 0
    for Xb, Tb, Yb in full_loader:
        Xb, Tb, Yb = Xb.to(DEVICE), Tb.to(DEVICE), Yb.to(DEVICE)
        opt.zero_grad()
        loss = crit(model_full(Xb, Tb, adj_t), Yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model_full.parameters(), 1.0)
        opt.step()
        total += loss.item() * Xb.size(0)
    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | loss: {total / len(full_ds):.4f}")

In [ ]:
from src import load_test_data, write_submission

test_hist, test_texts = load_test_data()
test_norm = normalize(test_hist, mean, std)
print("Encoding test texts...")
test_emb = encode_texts_qwen(test_texts, qwen_model, qwen_tokenizer,
                              DEVICE, batch_size=32)

@torch.no_grad()
def predict_test(model, X, T_emb, device, bs=16):
    model.eval()
    preds = []
    for i in range(0, len(X), bs):
        xb = torch.tensor(X[i:i+bs], dtype=torch.float32, device=device)
        tb = torch.tensor(T_emb[i:i+bs], dtype=torch.float32, device=device)
        pb = model(xb, tb, adj_t).cpu().numpy()
        preds.append(pb)
    yp_norm = np.concatenate(preds, axis=0)
    return denormalize(yp_norm, mean, std).clip(min=0)

test_preds = predict_test(model_full, test_norm, test_emb, DEVICE)
write_submission(test_preds, label="gnn_qwen")